# Final Dataset Validation

This notebook keeps only the checks needed before running the final experiments:

1. the data files can be loaded;
2. CVE rows have unique IDs, descriptions, and labels;
3. labels are clean and belong to the final CWE set;
4. every final class has at least 200 CVEs and the saved counts are consistent.


In [1]:
from collections import Counter
from pathlib import Path
import ast
import json
import re

import numpy as np
import pandas as pd


In [2]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

FILES = {
    "final_cve": DATA_DIR / "cve_dataset_hierarchy_merged.csv",
    "final_cwe": DATA_DIR / "cwe_hierarchy_merged_counts.csv",
}

MIN_SUPPORT = 200
EXPECTED_CLASSES = 150

missing_files = [path for path in FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing required files:" + "\n" + "\n".join(f"- {path}" for path in missing_files)
    )

df_cve = pd.read_csv(FILES["final_cve"], low_memory=False)
df_cwe = pd.read_csv(FILES["final_cwe"], low_memory=False)

print(f"Final CVEs: {len(df_cve):,}")
print(f"Final CWE classes: {len(df_cwe):,}")


Final CVEs: 291,935
Final CWE classes: 150


In [3]:
def find_column(df: pd.DataFrame, candidates: list[str]) -> str:
    normalized = {str(column).strip().lower(): column for column in df.columns}
    for candidate in candidates:
        key = candidate.strip().lower()
        if key in normalized:
            return normalized[key]
    raise KeyError(f"None of these columns were found: {candidates}")


def normalize_cwe_id(value: object) -> str | None:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    text = str(value).strip().upper()
    match = re.fullmatch(r"(?:CWE[-_ ]?)?(\d+)", text)
    if not match:
        return None
    return f"CWE-{int(match.group(1))}"


def parse_cwe_labels(value: object) -> list[str]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, (list, tuple, set, np.ndarray)):
        raw_items = list(value)
    else:
        text = str(value).strip()
        if not text:
            return []
        raw_items = None
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(text)
                if isinstance(parsed, (list, tuple, set)):
                    raw_items = list(parsed)
                    break
            except (ValueError, SyntaxError, TypeError, json.JSONDecodeError):
                pass
        if raw_items is None:
            raw_items = re.findall(r"CWE[-_ ]?\d+", text, flags=re.IGNORECASE)
    labels = [normalize_cwe_id(item) for item in raw_items]
    return sorted({label for label in labels if label is not None})


CVE_ID_COL = find_column(df_cve, ["cve_id", "cve", "id"])
DESCRIPTION_COL = find_column(df_cve, ["description", "summary", "text"])
LABELS_COL = find_column(df_cve, ["cwe_ids", "labels", "cwe"])
CWE_ID_COL = find_column(df_cwe, ["cwe_id", "cwe", "id"])
COUNT_COL = find_column(df_cwe, ["cve_count", "count", "support", "frequency"])

print("Detected columns:")
print(f"- CVE ID: {CVE_ID_COL}")
print(f"- Description: {DESCRIPTION_COL}")
print(f"- Labels: {LABELS_COL}")
print(f"- CWE ID: {CWE_ID_COL}")
print(f"- CWE count: {COUNT_COL}")


Detected columns:
- CVE ID: cve_id
- Description: description
- Labels: cwe_ids
- CWE ID: cwe_id
- CWE count: cve_count


## Validation Checks


In [4]:
df_cve = df_cve.copy()
df_cwe = df_cwe.copy()

df_cve["parsed_labels"] = df_cve[LABELS_COL].apply(parse_cwe_labels)
df_cve["n_labels"] = df_cve["parsed_labels"].str.len()
df_cwe[CWE_ID_COL] = df_cwe[CWE_ID_COL].apply(normalize_cwe_id)
df_cwe[COUNT_COL] = pd.to_numeric(df_cwe[COUNT_COL], errors="coerce")

final_classes = set(df_cwe[CWE_ID_COL].dropna())
all_labels = {label for labels in df_cve["parsed_labels"] for label in labels}
unknown_labels = sorted(all_labels - final_classes)
raw_label_text = df_cve[LABELS_COL].fillna("").astype(str)

recomputed_counts = Counter(
    label
    for labels in df_cve["parsed_labels"]
    for label in labels
)
recomputed = pd.DataFrame(
    recomputed_counts.items(),
    columns=[CWE_ID_COL, "recomputed_count"],
)
count_check = df_cwe[[CWE_ID_COL, COUNT_COL]].merge(
    recomputed,
    on=CWE_ID_COL,
    how="left",
)
count_check["recomputed_count"] = count_check["recomputed_count"].fillna(0)
count_check["difference"] = count_check["recomputed_count"] - count_check[COUNT_COL]

checks = pd.DataFrame(
    [
        {
            "check": "Required files loaded",
            "value": f"{len(df_cve):,} CVEs, {len(df_cwe):,} CWE rows",
            "passed": len(df_cve) > 0 and len(df_cwe) > 0,
        },
        {
            "check": "CVE rows are usable",
            "value": (
                f"duplicate IDs={df_cve[CVE_ID_COL].duplicated().sum()}, "
                f"empty descriptions={df_cve[DESCRIPTION_COL].fillna('').astype(str).str.strip().eq('').sum()}, "
                f"rows without labels={(df_cve['n_labels'] == 0).sum()}"
            ),
            "passed": (
                df_cve[CVE_ID_COL].duplicated().sum() == 0
                and df_cve[DESCRIPTION_COL].fillna('').astype(str).str.strip().ne('').all()
                and (df_cve["n_labels"] > 0).all()
            ),
        },
        {
            "check": "Labels are clean",
            "value": (
                f"unknown labels={len(unknown_labels)}, "
                f"NVD pseudo-label rows="
                f"{raw_label_text.str.contains('NVD-CWE-noinfo|NVD-CWE-Other', case=False, regex=True).sum()}"
            ),
            "passed": (
                len(unknown_labels) == 0
                and raw_label_text.str.contains(
                    "NVD-CWE-noinfo|NVD-CWE-Other",
                    case=False,
                    regex=True,
                ).sum() == 0
            ),
        },
        {
            "check": "Class support is valid",
            "value": (
                f"classes={len(final_classes)}, "
                f"min support={int(count_check[COUNT_COL].min())}, "
                f"count mismatches={(count_check['difference'] != 0).sum()}"
            ),
            "passed": (
                len(final_classes) == EXPECTED_CLASSES
                and count_check[COUNT_COL].min() >= MIN_SUPPORT
                and (count_check["difference"] == 0).all()
            ),
        },
    ]
)
checks["status"] = checks["passed"].map({True: "PASS", False: "FAIL"})
checks


,check,value,passed,status
0,Required files loaded,"291,935 CVEs, 150 CWE rows",True,PASS
1,CVE rows are usable,"duplicate IDs=0, empty descriptions=0, rows wi...",True,PASS
2,Labels are clean,"unknown labels=0, NVD pseudo-label rows=0",True,PASS
3,Class support is valid,"classes=150, min support=200, count mismatches=0",True,PASS


## Small Dataset Summary


In [5]:
summary = pd.DataFrame(
    {
        "metric": [
            "CVEs",
            "CWE classes",
            "CVE-CWE associations",
            "Average labels per CVE",
            "Multilabel CVEs (%)",
        ],
        "value": [
            len(df_cve),
            len(final_classes),
            int(df_cve["n_labels"].sum()),
            round(float(df_cve["n_labels"].mean()), 4),
            round(float((df_cve["n_labels"] > 1).mean() * 100), 2),
        ],
    }
)
summary


,metric,value
0,CVEs,291935.0000
1,CWE classes,150.0000
2,CVE-CWE associations,322946.0000
3,Average labels per CVE,1.1062
4,Multilabel CVEs (%),9.9900
